<a href="https://colab.research.google.com/github/elijahschoeniger-cpu/Walmart-Project-1-/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd, numpy as np

!git clone https://github.com/elijahschoeniger-cpu/Walmart-Project-1-.git

Cloning into 'Walmart-Project-1-'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 105 (delta 47), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 837.95 KiB | 6.76 MiB/s, done.
Resolving deltas: 100% (47/47), done.


### The data you are getting

Five files, in the `data/` folder next to this notebook.

| File | What it is | Size |
|---|---|---|
| `walmart_transactions.csv` | Point-of-sale records from Florida stores, September 2024 | about 25,000 rows |
| `store_locations.csv` | Store details: city, coordinates, region | about 14 rows |
| `product_catalog.csv` | The master list of products and SKUs | about 45 rows |
| `distribution_centers.csv` | Distribution center capacity and shipping cost | 5 rows |
| `inventory_and_sales.db` | SQLite database with `daily_sales_summary` and `inventory_levels` | about 7,900 rows across both tables |

Every file has quality problems, and finding them is part of the work. The brief lists what to expect:

- duplicate records
- missing values written several ways: empty, `NULL`, `N/A`, NaN
- dates and times in different formats
- the same product under several SKUs and names
- inconsistent capitalization and spacing
- category names that don't match
- numbers stored as text, and prices with `$` signs
- negative quantities
- store ids written different ways across files
- weather data in a different timezone than the sales data

You will profile the transactions file in a few minutes. The database matters mostly for Question 5.

In [5]:
import sqlite3

DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

# Get data from repository
REPO_PATH = '/content/Walmart-Project-1-/'
catalog_data = pd.read_csv(REPO_PATH + 'data/product_catalog.csv')
distribution_center = pd.read_csv(REPO_PATH + 'data/distribution_centers.csv')
walmart_transactions = pd.read_csv(REPO_PATH + 'data/walmart_transactions.csv')
store_locations = pd.read_csv(REPO_PATH + 'data/store_locations.csv')

# Read db into pandas
conn = sqlite3.connect('inventory_and_sales.db')
cursor = conn.cursor()

print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))
pd.read_sql_query('PRAGMA table_info(inventory_levels)', conn)

Empty DataFrame
Columns: [name]
Index: []


,cid,name,type,notnull,dflt_value,pk


Section for Cleaning (include decisions made using log)

In [6]:
import pandas as pd, os

def profile(df, name='dataset'):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    print()
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print()
    print('exact duplicate rows:', df.duplicated().sum())
    return summary

# %% [0] Setup — FIX for the empty database + reload raw data
import os, re, sqlite3
import numpy as np
import pandas as pd

REPO_PATH = globals().get('REPO_PATH', '/content/Walmart-Project-1-/')
DATA = os.path.join(REPO_PATH, 'data')

conn = sqlite3.connect(os.path.join(DATA, 'inventory_and_sales.db'))
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))

DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

# Read everything as text so pandas doesn't guess types; we convert explicitly.
NA_TOKENS = ['', 'NULL', 'N/A', 'NaN', 'nan', 'NA', 'None']
raw_txn = pd.read_csv(os.path.join(DATA, 'walmart_transactions.csv'), dtype=str,
                      keep_default_na=False)
raw_stores  = pd.read_csv(os.path.join(DATA, 'store_locations.csv'), dtype=str, keep_default_na=False)
raw_catalog = pd.read_csv(os.path.join(DATA, 'product_catalog.csv'), dtype=str, keep_default_na=False)
raw_dc      = pd.read_csv(os.path.join(DATA, 'distribution_centers.csv'), dtype=str, keep_default_na=False)
raw_sales   = pd.read_sql_query('SELECT * FROM daily_sales_summary', conn)
raw_inv     = pd.read_sql_query('SELECT * FROM inventory_levels', conn)
profile(raw_txn, 'walmart_transactions')
profile(raw_stores, 'store_locations')

# %% [1] Shared helpers + lookup tables
def to_na(s):
    """Strip whitespace and turn every missing-value spelling into a real NaN."""
    s = s.astype(str).str.strip()
    return s.mask(s.isin(NA_TOKENS))

def norm_store_id(s):
    """' FL-330', 'fl_330', 'FL330' -> 'FL-330'."""
    return (s.astype(str).str.strip().str.upper()
             .str.replace(r'^FL[\s_-]?(\d{3})$', r'FL-\1', regex=True))

# 45 SKUs collapse to 20 real products (verified: every SKU's names are variants of one item)
SKU_TO_PRODUCT = {
    'PT-12': 'PT-STRAW', 'SKU#459812': 'PT-STRAW', 'POPTART-STRAW': 'PT-STRAW', 'PT_STRAWBERRY': 'PT-STRAW',
    'WTR-24': 'WATER-24', 'WATER24PK': 'WATER-24', 'WTR_24PK': 'WATER-24',
    'BATT-100': 'BATT-AA4', 'BAT-AA4': 'BATT-AA4', 'BATT_AA_4PK': 'BATT-AA4',
    'FLASH-01': 'FLASHLIGHT', 'LED-FLASH': 'FLASHLIGHT',
    'BEER-24': 'BEER-24', 'LTBEER24': 'BEER-24',
    'BG-MONO': 'MONOPOLY', 'MONOPOLY-01': 'MONOPOLY', 'GAME-MONO': 'MONOPOLY',
    'CARDS-01': 'CARDS', 'PLAYCARD-STD': 'CARDS',
    'STREAM-01': 'STREAMING', 'ROKU-STD': 'STREAMING',
    'SOUP-CAN': 'SOUP', 'CNSOUP-01': 'SOUP',
    'BREAD-WH': 'BREAD', 'WHITEBREAD': 'BREAD',
    'PB-16': 'PB-16', 'PNUTBTR-16': 'PB-16',
    'CNDL-EM': 'CANDLES', 'CANDLE-EMRG': 'CANDLES',
    'FAID-01': 'FIRSTAID', 'FIRSTAID-KIT': 'FIRSTAID',
    'CHRG-PORT': 'CHARGER', 'PORTCHARGER': 'CHARGER',
    'CHIPS-VP': 'CHIPS-VP', 'CHIPVARIETY': 'CHIPS-VP',
    'PTOWEL-6': 'PTOWEL-6', 'PAPERTOWEL6': 'PTOWEL-6',
    'TP-12': 'TP-12', 'TOILETPAPER12': 'TP-12',
    'DIAP-S3': 'DIAPERS-3', 'DIAPERS-3': 'DIAPERS-3',
    'BFORM-01': 'FORMULA', 'BABYFORMULA': 'FORMULA',
    'TARP-HD': 'TARP', 'HDTARP-01': 'TARP',
}
PRODUCTS = pd.DataFrame([
    ('PT-STRAW',  'Pop-Tarts Strawberry',     'Breakfast'),
    ('WATER-24',  'Bottled Water 24-Pack',    'Emergency'),
    ('BATT-AA4',  'AA Batteries 4-Pack',      'Emergency'),
    ('FLASHLIGHT','LED Flashlight',           'Emergency'),
    ('CANDLES',   'Emergency Candles',        'Emergency'),
    ('TARP',      'Heavy Duty Tarp',          'Emergency'),
    ('BEER-24',   'Light Beer 24-Pack',       'Alcohol'),
    ('MONOPOLY',  'Monopoly Board Game',      'Entertainment'),
    ('CARDS',     'Playing Cards',            'Entertainment'),
    ('STREAMING', 'Roku Streaming Stick',     'Electronics'),
    ('CHARGER',   'Portable Phone Charger',   'Electronics'),
    ('SOUP',      'Canned Soup',              'Grocery'),
    ('BREAD',     'White Bread',              'Grocery'),
    ('PB-16',     'Peanut Butter 16oz',       'Grocery'),
    ('FIRSTAID',  'First Aid Kit',            'Health'),
    ('CHIPS-VP',  'Chips Variety Pack',       'Snacks'),
    ('PTOWEL-6',  'Paper Towels 6-Roll',      'Household'),
    ('TP-12',     'Toilet Paper 12-Roll',     'Household'),
    ('DIAPERS-3', 'Diapers Size 3',           'Baby'),
    ('FORMULA',   'Baby Formula',             'Baby'),
], columns=['product_id', 'product', 'category_clean'])

# Messy category text -> the same 10 categories the database already uses
CATEGORY_MAP = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks',
    'health': 'Health', 'health & wellness': 'Health',
}
def norm_category(s):
    return s.astype(str).str.strip().str.lower().map(CATEGORY_MAP)


# %% [2] Transactions
txn = raw_txn.copy()
n0 = len(txn)

# 2a. exact duplicates
dups = txn.duplicated().sum()
txn = txn.drop_duplicates().reset_index(drop=True)
log('txn_duplicates', 'Dropped exact duplicate rows (dup transaction_ids were all exact copies)', dups)
assert txn.transaction_id.is_unique

# 2b. store ids
before = txn.store_id.copy()
txn['store_id'] = norm_store_id(txn.store_id)
log('txn_store_id', "Normalized store_id to 'FL-###' (fixed case, spaces, '_' and missing dash)",
    (before != txn.store_id).sum())
assert txn.store_id.str.fullmatch(r'FL-\d{3}').all() and txn.store_id.nunique() == 12

# 2c. timestamps — 5 formats; parse each explicitly so nothing gets guessed wrong
ts = to_na(txn.timestamp)
n_missing_ts = ts.isna().sum()
FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
           '%m-%d-%Y %H:%M:%S', '%d-%b-%Y %H:%M']
parsed = pd.Series(pd.NaT, index=ts.index, dtype='datetime64[ns]')
for fmt in FORMATS:
    parsed = parsed.fillna(pd.to_datetime(ts, format=fmt, errors='coerce'))
unparsed = (ts.notna() & parsed.isna()).sum()
assert unparsed == 0, f'{unparsed} timestamps matched no known format'
txn['timestamp'] = parsed
log('txn_timestamp', 'Parsed 5 date formats (ISO, ISO-T, MM/DD/YYYY, MM-DD-YYYY, DD-Mon-YYYY) to datetime', len(txn) - n_missing_ts)
log('txn_timestamp_missing', "Missing timestamps (empty/NULL/N/A/NaN) set to NaT; rows KEPT for non-time questions, flagged ts_missing",
    n_missing_ts)
txn['ts_missing'] = txn.timestamp.isna()
assert txn.timestamp.dropna().between('2024-09-01', '2024-10-01').all()

# Timezone: POS times are local store time. Pensacola (FL-710) is in Central time; every other store is Eastern.
# timestamp_utc is what to join to the weather API data.
CENTRAL = {'FL-710'}
txn['tz'] = np.where(txn.store_id.isin(CENTRAL), 'America/Chicago', 'America/New_York')
txn['timestamp_utc'] = pd.concat([
    g.timestamp.dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')
    for tz, g in txn.groupby('tz')
]).sort_index()
log('txn_timezone', 'Localized timestamps (FL-710 Central, rest Eastern) and added timestamp_utc for the weather join',
    txn.timestamp_utc.notna().sum())
txn['date'] = txn.timestamp.dt.normalize()
txn['hour'] = txn.timestamp.dt.hour

# 2d. prices: '$8.26', ' 3.43 ' -> 8.26, 3.43
txn['unit_price'] = pd.to_numeric(txn.unit_price.str.replace(r'[\$,\s]', '', regex=True), errors='raise')
log('txn_unit_price', "Stripped '$' and whitespace from unit_price, cast to float",
    raw_txn.unit_price.str.contains(r'[\$\s]', regex=True).sum())

# 2e. quantity + negatives
txn['quantity'] = pd.to_numeric(txn.quantity, errors='raise').astype(int)
txn['is_return'] = txn.quantity < 0
log('txn_negative_qty', 'Negative quantities treated as returns: KEPT, flagged is_return. Demand analysis should filter quantity > 0',
    txn.is_return.sum())
txn['line_total'] = (txn.quantity * txn.unit_price).round(2)

# 2f. SKUs -> canonical product; categories from the product (not the messy text)
txn['sku'] = txn.sku.str.strip()
txn['product_id'] = txn.sku.map(SKU_TO_PRODUCT)
assert txn.product_id.notna().all(), 'unmapped SKU'
txn = txn.merge(PRODUCTS, on='product_id', how='left')
log('txn_sku_consolidation', f'Mapped {txn.sku.nunique()} SKUs / {raw_txn.product_name.nunique()} name variants -> {txn.product_id.nunique()} products',
    len(txn))

cat_from_text = norm_category(txn.category)
mismatch = (cat_from_text != txn.category_clean).sum()
assert cat_from_text.notna().all(), 'unmapped category spelling'
log('txn_category', f'Collapsed {raw_txn.category.nunique()} category spellings -> 10; product-derived category disagrees with text in {mismatch} rows',
    len(txn))
txn = txn.rename(columns={'product_name': 'product_name_raw', 'category': 'category_raw'})
txn = txn.rename(columns={'category_clean': 'category'})

print(f'\ntransactions: {n0:,} -> {len(txn):,} rows')
txn.head()


# %% [3] Store locations
stores = raw_stores.copy()
stores['store_id'] = norm_store_id(stores.store_id)
for c in stores.columns:
    stores[c] = to_na(stores[c])
dups = stores.duplicated('store_id').sum()
stores = stores.drop_duplicates('store_id').reset_index(drop=True)
log('stores_duplicates', "Dropped duplicate stores that only differed by id format (FL239, FL_287)", dups)

bad_state = (stores.state != 'FL').sum()
stores['state'] = 'FL'
log('stores_state', "Standardized state ('Florida', 'Fla.') -> 'FL'", bad_state)

stores['zip_code'] = stores.zip_code.str.zfill(5)
stores[['latitude', 'longitude']] = stores[['latitude', 'longitude']].astype(float)
miss = stores.latitude.isna()
# Tallahassee city-center coordinates — needed for the weather API call and the map
stores.loc[stores.store_id == 'FL-619', ['latitude', 'longitude']] = [30.4383, -84.2807]
log('stores_coords', 'Filled missing lat/lon for FL-619 with Tallahassee city-center coordinates (manual lookup)', miss.sum())

stores['region'] = stores.region.str.strip().str.title()
stores['tz'] = np.where(stores.store_id.isin(CENTRAL), 'America/Chicago', 'America/New_York')
assert stores.store_id.nunique() == 12 and set(stores.store_id) == set(txn.store_id)
stores


# %% [4] Product catalog
catalog = raw_catalog.copy()
for c in catalog.columns:
    catalog[c] = to_na(catalog[c])
catalog['product_id'] = catalog.sku.map(SKU_TO_PRODUCT)
assert catalog.product_id.notna().all()
catalog = catalog.merge(PRODUCTS, on='product_id', how='left')
catalog['category_raw'] = catalog.pop('category')
catalog = catalog.rename(columns={'category_clean': 'category', 'product_name': 'product_name_raw'})

catalog['unit_cost'] = pd.to_numeric(catalog.unit_cost)
miss = catalog.unit_cost.isna()
catalog['unit_cost_imputed'] = miss
catalog['unit_cost'] = catalog.unit_cost.fillna(catalog.groupby('product_id').unit_cost.transform('median'))
log('catalog_unit_cost', 'Missing unit_cost filled with median cost of the same product (other SKUs); flagged unit_cost_imputed', miss.sum())
log('catalog_supplier_pkg', 'supplier and package_size look random (e.g. Pop-Tarts from Roku Inc) — kept but marked unreliable, do not analyze', len(catalog))
catalog


# %% [5] Distribution centers
dc = raw_dc.copy()
for c in dc.columns:
    dc[c] = to_na(dc[c])
dc['surge_capacity_approx'] = dc.surge_capacity.str.contains('~', na=False)
dc['surge_capacity'] = pd.to_numeric(dc.surge_capacity.str.replace('~', '', regex=False))
log('dc_surge_capacity', "Removed '~' from surge_capacity ('~750' -> 750), flagged surge_capacity_approx", dc.surge_capacity_approx.sum())
for c in ['avg_daily_shipments', 'distance_to_orlando_mi', 'cost_per_shipment']:
    dc[c] = pd.to_numeric(dc[c])
log('dc_cost_missing', 'DC-BHM cost_per_shipment missing — left NaN (no defensible way to impute from 4 rows)',
    dc.cost_per_shipment.isna().sum())
dc


# %% [6] Database tables (these feed Q5 — flag oddities, don't delete them)
sales = raw_sales.copy()
compact = sales.date.str.fullmatch(r'\d{8}')
sales['date'] = pd.to_datetime(sales.date, format='%Y-%m-%d', errors='coerce').fillna(
                pd.to_datetime(sales.date, format='%Y%m%d', errors='coerce'))
assert sales.date.notna().all()
log('db_sales_date', "daily_sales_summary: dates in two formats ('2024-09-12' and '20240912') parsed to datetime", compact.sum())
sales['store_id'] = norm_store_id(sales.store_id)
sales['category'] = norm_category(sales.category)
assert sales.category.notna().all()
dups = sales.duplicated(['date', 'store_id', 'category']).sum()
sales = sales.drop_duplicates(['date', 'store_id', 'category']).reset_index(drop=True)
log('db_sales_duplicates', 'daily_sales_summary: duplicate (date, store, category) rows after date fix — dropped', dups)

sales['units_fractional'] = sales.total_units % 1 != 0
sales['revenue_negative'] = sales.total_revenue < 0
log('db_sales_fractional', 'daily_sales_summary: fractional total_units (impossible for unit counts) — flagged units_fractional, kept for Q5',
    sales.units_fractional.sum())
log('db_sales_negative', 'daily_sales_summary: negative total_revenue — flagged revenue_negative, kept for Q5',
    sales.revenue_negative.sum())

inv = raw_inv.copy()
inv['date'] = pd.to_datetime(inv.date)
inv['store_id'] = norm_store_id(inv.store_id)
inv['product_id'] = inv.sku.map(SKU_TO_PRODUCT)
orphans = inv.product_id.isna()
log('db_inv_orphan_skus', f'inventory_levels: SKUs not in catalog {sorted(inv.loc[orphans, "sku"].unique())} — dropped', orphans.sum())
inv = inv[~orphans].reset_index(drop=True)
log('db_inv_on_order', 'inventory_levels: units_on_order missing — left NaN (unknown is not the same as 0)',
    inv.units_on_order.isna().sum())
inv['below_reorder'] = inv.units_on_hand < inv.reorder_point


# %% [7] Final checks, save, decision log
assert txn.store_id.isin(stores.store_id).all()
assert sales.store_id.isin(stores.store_id).all() and inv.store_id.isin(stores.store_id).all()
assert set(txn.category) == set(sales.category)

OUT = os.path.join(DATA, 'clean')
os.makedirs(OUT, exist_ok=True)
txn.to_csv(os.path.join(OUT, 'transactions_clean.csv'), index=False)
stores.to_csv(os.path.join(OUT, 'stores_clean.csv'), index=False)
catalog.to_csv(os.path.join(OUT, 'catalog_clean.csv'), index=False)
PRODUCTS.to_csv(os.path.join(OUT, 'products.csv'), index=False)
dc.to_csv(os.path.join(OUT, 'distribution_centers_clean.csv'), index=False)
sales.to_csv(os.path.join(OUT, 'daily_sales_clean.csv'), index=False)
inv.to_csv(os.path.join(OUT, 'inventory_clean.csv'), index=False)
print('saved to', OUT)

pd.set_option('display.max_colwidth', None)
show_log()



                  name
0  daily_sales_summary
1     inventory_levels
=== walmart_transactions ===
25,562 rows x 8 columns

                 dtype  nulls  null_pct  distinct
transaction_id  object      0       0.0     25532
store_id        object      0       0.0        60
timestamp       object      0       0.0     23957
sku             object      0       0.0        45
product_name    object      0       0.0        63
category        object      0       0.0        37
quantity        object      0       0.0        11
unit_price      object      0       0.0      3668

exact duplicate rows: 30
=== store_locations ===
14 rows x 8 columns

             dtype  nulls  null_pct  distinct
store_id    object      0       0.0        14
store_name  object      0       0.0        12
city        object      0       0.0        12
state       object      0       0.0         3
zip_code    object      0       0.0        12
latitude    object      0       0.0        12
longitude   object      0       0.

,step,decision,rows
0,txn_duplicates,Dropped exact duplicate rows (dup transaction_ids were all exact copies),30
1,txn_store_id,"Normalized store_id to 'FL-###' (fixed case, spaces, '_' and missing dash)",4345
2,txn_timestamp,"Parsed 5 date formats (ISO, ISO-T, MM/DD/YYYY, MM-DD-YYYY, DD-Mon-YYYY) to datetime",24648
3,txn_timestamp_missing,"Missing timestamps (empty/NULL/N/A/NaN) set to NaT; rows KEPT for non-time questions, flagged ts_missing",884
4,txn_timezone,"Localized timestamps (FL-710 Central, rest Eastern) and added timestamp_utc for the weather join",24648
5,txn_unit_price,"Stripped '$' and whitespace from unit_price, cast to float",4332
6,txn_negative_qty,"Negative quantities treated as returns: KEPT, flagged is_return. Demand analysis should filter quantity > 0",130
7,txn_sku_consolidation,Mapped 45 SKUs / 63 name variants -> 20 products,25532
8,txn_category,Collapsed 37 category spellings -> 10; product-derived category disagrees with text in 0 rows,25532
9,stores_duplicates,"Dropped duplicate stores that only differed by id format (FL239, FL_287)",2


SKU Consolidation

In [4]:
# Work on consolidating the SKUs!

API Call Code

In [5]:
# Connect to open-meteo
import requests

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

# A saved copy of the same Open-Meteo data (Orlando, Sept 1-30, 2024) for when you have no wifi
# or the API is down. The source column says so.
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'fallback sample',
})

try:
    weather_live = daily_weather('2024-09-01', '2024-09-30')
except requests.RequestException as err:
    print('API call failed:', err)
    weather_live = FALLBACK

same = (np.allclose(weather_live['precipitation_sum'], FALLBACK['precipitation_sum'])
        and np.allclose(weather_live['windspeed_10m_max'], FALLBACK['windspeed_10m_max']))
print('source:', weather_live['source'].iloc[0])
print('matches the saved copy:', same)
weather_live.head()

source: open-meteo
matches the saved copy: True


,date,precipitation_sum,windspeed_10m_max,source
0,2024-09-01,3.0,18.8,open-meteo
1,2024-09-02,12.4,13.5,open-meteo
2,2024-09-03,24.9,22.8,open-meteo
3,2024-09-04,22.1,15.5,open-meteo
4,2024-09-05,23.1,15.2,open-meteo


In [6]:
# WAITING TO MERGE WITH DATA FROM CLEANED DATABASES

Section for Showing Log

In [7]:
show_log()

""
